In [1]:
import random, os
import numpy as np
import torch
os.environ["CUDA_VISIBLE_DEVICES"]="1"

from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, AutoModel
from datasets import load_dataset
import pandas as pd
import re

device1 = 'cuda:0'
device2 = 'cuda:1'
data_dir = '/raid/deallab/SF_RAG_Data/ASQA'
# data_dir = '../data'

/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed_value=42):
    # Set seed for reproducibility.
    random.seed(seed_value)
    os.environ['PYTHONHASHSEED']=str(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.backends.cudnn.deterministic=True    
    torch.backends.cudnn.benchmark=True
    torch.cuda.manual_seed_all(seed_value)

In [3]:
# set_seed(30)

In [3]:
#load embeddings
embedd_test_path = f'{data_dir}/test/embedd_test.npy'
evidence_embeddings = np.load(embedd_test_path)
print(evidence_embeddings.shape)
evidence_embeddings = torch.from_numpy(evidence_embeddings).to(device1)

#load evidence
evidence_test_path = f'{data_dir}/test/evidence_test.csv'
evidence_df = pd.read_csv(evidence_test_path)

#load qa data
qa_df=pd.read_csv(f'{data_dir}/test/qa_test.csv') #data=df[['question','long_answers']] # questions=data['question'] #references = [row.to_dict() for i, row in df.iterrows() if i < len(questions)]
qa_df.head()

(21586, 4096)


,id,sample_id,question,follow_up_questions,long_answers,short_answers
0,c2687961-0957-45cb-bae0-42314e38f790,-7013890438520559398,Who has the highest goals in world football?,"[""Who has the highest goals in men's world int...","[""Ali Dael has the highest goals in men's worl...","[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'..."
1,26830122-8240-40a9-aaff-d9731d53b197,7089015503030534342,Who is the original artist of sound of silence?,['Who is the original artist of sound of silen...,[' The original artist of the song sound of si...,"[['Simon & Garfunkel', 'Paul Simon and Art Gar..."
2,268116a9-5ecb-4364-8da4-4a648f9d5b43,8793099883447006698,When was the first apple i phone made?,"['When was the first apple i phone released?',...",['The iPhone beta was created in 2004 to test ...,"[['June 29, 2007'], ['2004'], ['June 29, 2007...."
3,efb4810e-637b-4954-a776-3c2d05d1290c,-881464876144297194,Who played the weasley brothers in harry potter?,['Who played Bill weasley in Harry Potter and...,['Rupert Grint played Ron Weasley in all the H...,"[['Richard Fish'], ['Chris Rankin'], ['James P..."
4,99817eba-d32a-4c4d-9fe2-93a50ae1d367,1650309494326541834,How many state parks are there in virginia?,['How many state parks are there in virginia i...,['When the Virginia state park system was form...,"[['six'], ['38'], ['6'], ['38']]"


In [4]:
#load quantized model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

# load model with tokenizer
model = AutoModel.from_pretrained(
    'nvidia/NV-Embed-v2', 
    trust_remote_code=True,
    quantization_config = bnb_config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage =True,
)
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:08<00:00,  2.04s/it]


NVEmbedModel(
  (latent_attention_model): LatentAttentionModel(
    (cross_attend_blocks): ModuleList(
      (0): PreNorm(
        (fn): Attention(
          (to_q): Linear4bit(in_features=4096, out_features=32768, bias=False)
          (to_kv): Linear4bit(in_features=4096, out_features=65536, bias=False)
          (to_out): Linear4bit(in_features=32768, out_features=4096, bias=False)
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (norm_context): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
      (1): PreNorm(
        (fn): FeedForward(
          (net): Sequential(
            (0): Linear4bit(in_features=4096, out_features=32768, bias=True)
            (1): GEGLU()
            (2): Linear4bit(in_features=16384, out_features=4096, bias=True)
          )
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (embedding_model): BidirectionalMistralModel(
    (embed_tokens): Embedding(

In [5]:
#load tokenizer
tokenizer_gen = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
# cache_dir= '/raid/deallab/.cache')
tokenizer_gen.pad_token = tokenizer_gen.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    # bnb_4bit_quant_type="nf4",
    # bnb_4bit_compute_dtype=torch.bfloat16,
    # bnb_4bit_use_double_quant=True,
    # bnb_4bit_quant_storage=torch.bfloat16,
)

model_gen = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map= 'auto',
    # cache_dir= '/raid/deallab/.cache'
)
model_gen.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:17<00:00,  4.26s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): Ll

In [5]:
# import pandas as pd
# evidence_test_path = f'/raid/deallab/SF_RAG_Data/ASQA/test/evidence_test.csv'

# evidence_text = pd.read_csv(evidence_test_path)

In [8]:
# evidence_text_list = evidence_text['text'].tolist()

In [6]:
# from langchain_text_splitters import TokenTextSplitter

# text_splitter = TokenTextSplitter(
#     chunk_size=500,  # 청크 크기를 10으로 설정합니다.
#     chunk_overlap=50,  # 청크 간 중복을 0으로 설정합니다.
# )
# # combined_text = " ".join(evidence_text_list)
# # texts = text_splitter.split_text(combined_text)
# split_texts = [text_splitter.split_text(text)[0] for text in evidence_text_list]
# print(split_texts[0])

In [72]:
# from langchain.retrievers import BM25Retriever, EnsembleRetriever
# from langchain.vectorstores import FAISS

# # bm25 retriever와 faiss retriever를 초기화합니다.
# bm25_retriever = BM25Retriever.from_texts(
#     evidence_text_list,
# )
# bm25_retriever.k = 10  # BM25Retriever의 검색 결과 개수를 1로 설정합니다.

# embedding = model
# faiss_vectorstore = FAISS.from_texts(
#     evidence_text,
#     embedding,
# )
# faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 2})

# # 앙상블 retriever를 초기화합니다.
# ensemble_retriever = EnsembleRetriever(
#     retrievers=[bm25_retriever, faiss_retriever],
#     weights=[0.7, 0.3],
# )

In [75]:
# from langchain_community.document_transformers import LongContextReorder

# def bm25_retrieve(query):
#     bm25_result = bm25_retriever.invoke(query)
#     bm25_docs=list()

#     print("[BM25 Retriever]")
#     for doc in bm25_result:
#         # print(f"Content: {doc.page_content}")
#         # print()
#         bm25_docs.append(doc.page_content)
#     reordering = LongContextReorder()
#     bm25_docs = reordering.transform_documents(bm25_docs)
#     return bm25_docs

In [7]:
# res=bm25_retrieve("Who has the highest goals in world football?")
# res

In [6]:
# retrive docs from the document embeddings
def retrieve_documents(query):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)
    
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:10].cpu().detach().numpy()
    res=[evidence_df.loc[idx, 'text'] for idx in top_results if idx < len(evidence_df)]
        
    return res

In [7]:
def summarize(query, docs):
    prompt = """
    In a Retrieval Augmentation Generation system, documents close to the query vector are as follows:
    ---------------------
    {0}
    ---------------------
    Identify entities and contexts in a query, and use them to extract and summarize only relevant content from documents.
    Query: {1}
    Answer:
    """.format('\n'.join(docs), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device2)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device2)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [11]:
def various_answer(query, docs):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    There may be multiple golden short answers in your answers, and they should be explained.
    Query: {1}
    Answer:
    """.format('\n'.join(docs), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [9]:
def various_answer(query, docs, first_ans=None):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    There may be multiple golden short answers in your answers, and they should be explained.
    Query: {1}{2}
    Answer:
    """.format('\n'.join(docs), query, f"Prior Answer: {first_ans}")
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [12]:
def answer(query, context):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    Query: {1}
    Answer:
    """.format('\n'.join(context), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [43]:
# def HyDE(query, docs):
#     prompt = """
#     In a Retrieval Augmentation Generation system, documents close to the query vector are as follows:
#     ---------------------
#     {0}
#     ---------------------
#     Identify entities and contexts in a query, and use them to extract and summarize only relevant content from documents.
#     Query: {1}
#     Answer:
#     """.format('\n'.join(docs), query)
#     input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device2)

#     attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device2)

#     out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
#     res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
#     return [re.sub('\n|<\|eot_id\|>', '', res)]

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-reranker-v2-m3')
# model = AutoModelForSequenceClassification.from_pretrained('BAAI/bge-reranker-v2-m3')
# model.eval()

# with torch.no_grad():
#     inputs = tokenizer(pairs, padding=True, truncation=True, return_tensors='pt', max_length=512)
#     scores = model(**inputs, return_dict=True).logits.view(-1, ).float()
#     scores = exp_normalize(scores.numpy()) 
    
# print(np.round(scores * 100, 2))

# Baseline

In [13]:
from tqdm import tqdm
from evaluation import evaluate

stop_iteration = 20

scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    retrieved_docs = retrieve_documents(query)
    ans=answer(query,retrieved_docs)
    print('Final ans:', ans)
    scores=evaluate(ans, [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]/home/dataconv/.cache/huggingface/modules/transformers_modules/nvidia/NV-Embed-v2/5130cf1daf847c1bacee854a6ef1ca939e747fb2/modeling_nvembed.py:349: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(batch_dict.get('input_ids').to(batch_dict.get('input_ids')).long()),
/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Final ans: ['Based on the provided context information, the player with the highest goals in world football is Ali Daei of Iran, who has scored 109 international goals.']
Based on the provided context information, the player with the highest goals in world football is Ali Daei of Iran, who has scored 109 international goals.
Who has the highest goals in world football?
["Who has the highest goals in men's world international football?", "Who has the highest goals all-time in men's football?", "Who has the highest goals in women's world international football?"]
[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'], ['Sinclair', 'Christine Sinclair']]


  5%|▌         | 1/20 [00:04<01:28,  4.65s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.8181055784225464, 'start': 98, 'end': 106, 'answer': 'Ali Daei'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.1297767013311386, 'start': 98, 'end': 106, 'answer': 'Ali Daei'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 0.026729585602879524, 'start': 98, 'end': 106, 'answer': 'Ali Daei'}
{'rougeLsum': 30.303030303030305, 'length': 26.0, 'str_em': 33.33333333333333, 'Disambig-F1': 33.33333333333333}
Final ans: ['The original artist of "The Sound of Silence" is Simon & Garfunkel.']
The original artist of "The Sound of Silence" is Simon & Garfunkel.
Who is the original artist of sound of silence?
['Who is the original artist of sound of silence, the song, released in 1964?', 'Who is t

 10%|█         | 2/20 [00:07<01:04,  3.58s/it]

...

 95%|█████████▌| 19/20 [01:08<00:03,  3.81s/it]

follow question : Who plays patrick in  the 1999 film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger', 'Ledger']
{'score': 0.9782969951629639, 'start': 89, 'end': 101, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the 2009 tv series 10 things i hate about you?
short answer : ['Ethan Peck', 'Peck', 'Ethan Gregory Peck']
{'score': 0.9228044748306274, 'start': 210, 'end': 220, 'answer': 'Ethan Peck'}
follow question : Who plays patrick in the film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger']
{'score': 0.9191804528236389, 'start': 89, 'end': 101, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the TV series 10 things i hate about you?
short answer : ['Ethan Peck', 'Ethan Gregory Peck']
{'score': 0.9553555846214294, 'start': 210, 'end': 220, 'answer': 'Ethan Peck'}
{'rougeLsum': 67.5, 'length': 39.0, 'str_em': 100.0, 'Disambig-F1': 100.0}
Final ans: ['No, Microsoft Live Movie Maker is

100%|██████████| 20/20 [01:13<00:00,  3.67s/it]

follow question : Microsoft live movie maker is an example of a freely licensed software, often called free what?
short answer : ['freeware']
{'score': 0.20378831028938293, 'start': 66, 'end': 74, 'answer': 'freeware'}
follow question : Microsoft live movie maker is an example of free software used for what purpose?
short answer : ['Video editing software']
{'score': 0.039493441581726074, 'start': 66, 'end': 74, 'answer': 'freeware'}
{'rougeLsum': 37.2093023255814, 'length': 51.0, 'str_em': 50.0, 'Disambig-F1': 50.0}


rougeLsum      31.327766
length         23.650000
str_em         47.916667
Disambig-F1    45.589286
dtype: float64

# Answer RAG

In [14]:
from tqdm import tqdm
from evaluation import evaluate

# set_seed(30)

stop_iteration = 20

scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    retrieved_docs = retrieve_documents(query)
    first_ans=various_answer(query,retrieved_docs)
    print('First ans:', first_ans)
    ans_docs=retrieve_documents(first_ans)
    final_ans=various_answer(query,ans_docs)
    print('Second ans:', final_ans)
    scores=evaluate([final_ans], [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]

First ans: The answer to this query is Ali Daei of Iran, who has scored 109 goals in international football, making him the highest goalscorer in the world.Explanation:Ali Daei holds the record for most goals scored in international football with 109 goals. He achieved this feat in 149 matches for Iran, with a goal-scoring ratio of 0.73 goals per match.Golden Short Answer: Ali Daei is the highest goalscorer in the world with 109 goals.This answer is based on the information provided in the context, specifically from the document "List of players" where Ali Daei is listed as the second-highest international goalscorer with 109 goals.
Second ans: Josef Bican holds the record for the highest goals in world football, with a total of 805 goals in his career. However, it's essential to note that this record is based on the data available from the Rec.Sport.Soccer Statistics Foundation, which may not be comprehensive or up-to-date.Josef Bican played from 1928 to 1955, and his goal-scoring rat

  5%|▌         | 1/20 [00:18<05:49, 18.41s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.9616044163703918, 'start': 0, 'end': 11, 'answer': 'Josef Bican'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.09467499703168869, 'start': 0, 'end': 11, 'answer': 'Josef Bican'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 2.883006970932911e-07, 'start': 0, 'end': 11, 'answer': 'Josef Bican'}
{'rougeLsum': 30.769230769230777, 'length': 105.0, 'str_em': 33.33333333333333, 'Disambig-F1': 33.33333333333333}
First ans: The original artist of "The Sound of Silence" is Simon & Garfunkel, an American musical duo composed of Paul Simon and Art Garfunkel.
Second ans: The original artist of "The Sound of Silence" is Simon & Garfunkel, an American musical duo composed of Paul Simon and Art Ga

 10%|█         | 2/20 [00:25<03:28, 11.57s/it]

...

 95%|█████████▌| 19/20 [04:02<00:14, 14.46s/it]

follow question : Who plays patrick in  the 1999 film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger', 'Ledger']
{'score': 0.9443969130516052, 'start': 75, 'end': 87, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the 2009 tv series 10 things i hate about you?
short answer : ['Ethan Peck', 'Peck', 'Ethan Gregory Peck']
{'score': 0.8853476047515869, 'start': 181, 'end': 191, 'answer': 'Ethan Peck'}
follow question : Who plays patrick in the film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger']
{'score': 0.7722824215888977, 'start': 75, 'end': 87, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the TV series 10 things i hate about you?
short answer : ['Ethan Peck', 'Ethan Gregory Peck']
{'score': 0.9444693922996521, 'start': 181, 'end': 191, 'answer': 'Ethan Peck'}
{'rougeLsum': 50.48543689320388, 'length': 59.0, 'str_em': 100.0, 'Disambig-F1': 100.0}
First ans: Microsoft Live Movie Mak

100%|██████████| 20/20 [04:14<00:00, 12.73s/it]

follow question : Microsoft live movie maker is an example of a freely licensed software, often called free what?
short answer : ['freeware']
{'score': 0.4661667048931122, 'start': 186, 'end': 194, 'answer': 'freeware'}
follow question : Microsoft live movie maker is an example of free software used for what purpose?
short answer : ['Video editing software']
{'score': 0.9100143313407898, 'start': 56, 'end': 69, 'answer': 'video editing'}
{'rougeLsum': 36.19047619047619, 'length': 69.0, 'str_em': 100.0, 'Disambig-F1': 90.0}


rougeLsum      37.282190
length         54.900000
str_em         57.500000
Disambig-F1    51.896825
dtype: float64

In [15]:
scores_df.to_csv('./results/answer_rag_3_results.csv', index=False)

In [16]:
import pandas as pd
import math
sf = pd.read_csv('results/answer_rag_3_results.csv')
sf=sf[sf['length']<1000][:100]
print(len(sf))
print(sf.mean())
r=sf.mean()['rougeLsum']
d=sf.mean()['Disambig-F1']
dr=math.sqrt(r*d)
print(dr)

20
rougeLsum      37.282190
length         54.900000
str_em         57.500000
Disambig-F1    51.896825
dtype: float64
43.9866718486778


In [27]:
from tqdm import tqdm
from evaluation import evaluate

set_seed()

stop_iteration = 20

scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    retrieved_docs = retrieve_documents(query)
    first_ans=answer(query,retrieved_docs)
    print('First ans:', first_ans[0])
    ans_docs=retrieve_documents(first_ans[0])
    final_ans=answer(first_ans[0],ans_docs)
    print('Second ans:', final_ans[0])
    scores=evaluate(final_ans, [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]/home/dataconv/.cache/huggingface/modules/transformers_modules/nvidia/NV-Embed-v2/5130cf1daf847c1bacee854a6ef1ca939e747fb2/modeling_nvembed.py:349: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(batch_dict.get('input_ids').to(batch_dict.get('input_ids')).long()),
/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


First ans: According to the provided context information, the player with the highest goals in world football is Ali Daei of Iran, with 109 goals in international matches.
Second ans: According to the provided context information, the player with the highest goals in world football is Ali Daei of Iran, with 109 goals in international matches.
According to the provided context information, the player with the highest goals in world football is Ali Daei of Iran, with 109 goals in international matches.
Who has the highest goals in world football?
["Who has the highest goals in men's world international football?", "Who has the highest goals all-time in men's football?", "Who has the highest goals in women's world international football?"]
[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'], ['Sinclair', 'Christine Sinclair']]


  5%|▌         | 1/20 [00:07<02:27,  7.74s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.8568584322929382, 'start': 102, 'end': 110, 'answer': 'Ali Daei'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.17466114461421967, 'start': 102, 'end': 110, 'answer': 'Ali Daei'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 0.05843370407819748, 'start': 102, 'end': 110, 'answer': 'Ali Daei'}
{'rougeLsum': 36.36363636363637, 'length': 26.0, 'str_em': 33.33333333333333, 'Disambig-F1': 33.33333333333333}
First ans: Simon & Garfunkel
Second ans: Simon & Garfunkel was an American folk rock duo consisting of Paul Simon and Art Garfunkel. They were one of the most popular and influential musical acts of the 1960s, known for their harmonious vocals and introspective songwriting.Simon & Garf

 10%|█         | 2/20 [00:36<06:06, 20.37s/it]

...

 95%|█████████▌| 19/20 [03:06<00:09,  9.98s/it]

follow question : Who plays patrick in  the 1999 film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger', 'Ledger']
{'score': 0.9635234475135803, 'start': 68, 'end': 80, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the 2009 tv series 10 things i hate about you?
short answer : ['Ethan Peck', 'Peck', 'Ethan Gregory Peck']
{'score': 1.869488914962858e-05, 'start': 68, 'end': 80, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger']
{'score': 0.8692940473556519, 'start': 68, 'end': 80, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the TV series 10 things i hate about you?
short answer : ['Ethan Peck', 'Ethan Gregory Peck']
{'score': 0.00026919867377728224, 'start': 68, 'end': 80, 'answer': 'Heath Ledger'}
{'rougeLsum': 42.10526315789474, 'length': 35.0, 'str_em': 50.0, 'Disambig-F1': 50.0}
First ans: No, Microsoft Live 

100%|██████████| 20/20 [03:14<00:00,  9.75s/it]

follow question : Microsoft live movie maker is an example of a freely licensed software, often called free what?
short answer : ['freeware']
{'score': 0.023067617788910866, 'start': 55, 'end': 63, 'answer': 'freeware'}
follow question : Microsoft live movie maker is an example of free software used for what purpose?
short answer : ['Video editing software']
{'score': 0.2880362868309021, 'start': 239, 'end': 255, 'answer': 'download and use'}
{'rougeLsum': 32.608695652173914, 'length': 56.0, 'str_em': 50.0, 'Disambig-F1': 50.0}


rougeLsum      33.400839
length         61.800000
str_em         48.333333
Disambig-F1    41.194444
dtype: float64

In [25]:
scores_df.to_csv('./results/answer_rag_2_results.csv', index=False)

In [26]:
import pandas as pd
import math
sf = pd.read_csv('results/answer_rag_2_results.csv')
sf=sf[sf['length']<1000][:100]
print(len(sf))
print(sf.mean())
r=sf.mean()['rougeLsum']
d=sf.mean()['Disambig-F1']
dr=math.sqrt(r*d)
print(dr)

16
rougeLsum      33.482502
length         97.562500
str_em         62.500000
Disambig-F1    42.708333
dtype: float64
37.815101059628596
